# Ejemplo avanzado de PySpark
Este notebook demuestra agregaciones, funciones de ventana, análisis, particiones y operaciones Delta Lake.

In [ ]:
!pip install -q pyspark delta-spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (SparkSession.builder.master('local[*]')
         .appName('PySparkAdvanced')
         .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
         .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
         .getOrCreate())

## 1. Preparar datos de ejemplo

In [ ]:
data = [
    (1, 'A', 10, '2024-01-01'),
    (2, 'A', 15, '2024-01-03'),
    (3, 'B', 20, '2024-01-07'),
    (4, 'B', 25, '2024-01-09'),
    (5, 'C', 30, '2024-01-10')
]
columns = ['id', 'categoria', 'valor', 'fecha']
df = spark.createDataFrame(data, columns)
df = df.withColumn('fecha', F.to_date('fecha'))
df.show()

## 2. Agregación de datos

In [ ]:
agg_df = df.groupBy('categoria').agg(
    F.count('*').alias('conteo'),
    F.sum('valor').alias('suma_valor'),
    F.avg('valor').alias('promedio_valor')
)
agg_df.show()

## 3. Uso de funciones de ventana

In [ ]:
window_spec = Window.partitionBy('categoria').orderBy('fecha')
df = df.withColumn('rn', F.row_number().over(window_spec))
df = df.withColumn('running_total', F.sum('valor').over(window_spec))
df.show()

## 4. Análisis de datos

In [ ]:
df.describe(['valor']).show()
top_valor = df.orderBy(F.desc('valor')).limit(1)
top_valor.show()

## 5. Particionamiento de datos

In [ ]:
path = '/content/delta_table'
df.write.format('delta').mode('overwrite').partitionBy('categoria').save(path)

## 6. Partition pruning

In [ ]:
df_pruned = spark.read.format('delta').load(path).filter(F.col('categoria') == 'A')
df_pruned.explain(True)
df_pruned.show()

## 7. Delta merge

In [ ]:
from delta.tables import DeltaTable
new_data = [(2, 'A', 18, '2024-01-04'), (6, 'C', 40, '2024-01-11')]
updates = spark.createDataFrame(new_data, columns)
updates = updates.withColumn('fecha', F.to_date('fecha'))
delta_table = DeltaTable.forPath(spark, path)
delta_table.alias('t').merge(
    updates.alias('s'),
    't.id = s.id'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
delta_table.toDF().show()

In [ ]:
spark.stop()